# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains comprehensive tabular data of 77 cancer survivors with second primary colorectal cancer. Variables include demographic, clinicopathological, and molecular biomarker information.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset object from the schema URL
dataset = mlc.Dataset(croissant_url)

# The metadata attribute provides a high-level summary
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Number of record sets found: {len(dataset.metadata.record_sets)}")

## 2. Data Overview

Review available record sets, their fields, and associated `@id`s.

The `@id` fields uniquely identify record sets, fields, and columns. We'll print out all available record sets and their schemas.

In [ ]:
# List all record sets in the dataset by @id and name
record_sets_info = []
for rs in dataset.metadata.record_sets:
    print(f"RecordSet @id: {rs.id}\n  name: {rs.name}\n  description: {rs.description}")
    for field in rs.fields:
        print(f"    Field @id: {field.id}, name: {field.name}, type: {field.data_type}")
    print('-'*60) 
    record_sets_info.append({'id': rs.id, 'name': rs.name})

Below we load and print a few records from each available record set (using their `@id`):

In [ ]:
# For demonstration, print up to 2 sample records per record set
for rs in dataset.metadata.record_sets:
    print(f"--- {rs.name} (@id: {rs.id}) ---")
    try:
        iterator = dataset.records(record_set=rs.id)
        for i, record in enumerate(iterator):
            if i >= 2:
                break
            print(record)
    except Exception as e:
        print(f"Could not load records for {rs.id}: {e}")
    print()

## 3. Data Extraction

Let's extract tabular records from the primary record set(s) using their `@id` and load them into pandas DataFrames for further analysis.

In [ ]:
# Create DataFrames for each record set for easy EDA
dataframes = {}

for rs in dataset.metadata.record_sets:
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Loaded {len(df)} records for RecordSet '{rs.name}' (@id: {rs.id})")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not extract DataFrame for {rs.id}: {e}")

## 4. Exploratory Data Analysis (EDA)

We select a numeric field (`@id`) within the primary record set for EDA. Example operations performed below: filter, normalization, and grouping by a categorical field (`@id`).

_Please update the `numeric_field_id` and `group_field_id` to match the actual @id values from the previous overview when running interactively. This notebook uses an example based on the dataset content and prints available options._

In [ ]:
# Select the main record set (choose the first one as an example)
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]
    print(f"Available columns in '{main_record_set_id}':\n", df.columns.tolist())

    # Let's choose as numeric_field_id any column with numeric dtype (example: 'Age')
    numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    print('Numeric candidate fields:', numeric_columns)

    if numeric_columns:
        numeric_field_id = numeric_columns[0]  # Use the first numeric field
        threshold = df[numeric_field_id].mean()  # Use the mean as an example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try to find a suitable group-by field (categorical)
        candidate_groups = df.select_dtypes(include=['object']).columns.tolist()
        print('Categorical/group fields:', candidate_groups)
        if len(candidate_groups) > 0:
            group_field_id = candidate_groups[0]
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
                print(f"Mean {numeric_field_id} by {group_field_id}:")
                display(grouped_df.head())
            else:
                print(f"Group field {group_field_id} not in DataFrame columns.")
        else:
            print("No categorical fields available for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No DataFrames loaded for EDA.")

## 5. Visualization

We can visualize the distribution of a numeric field or relationship between key features. Below is an example histogram and a group mean bar plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # Bar plot for group mean
    if 'grouped_df' in locals() and grouped_df.shape[1] == 2:
        plt.figure(figsize=(9,4))
        sns.barplot(data=grouped_df, x=grouped_df.columns[0], y=grouped_df.columns[1])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we have:
- Loaded the FAIR^2 colorectal cancer dataset via its Croissant schema using `mlcroissant`.
- Explored the record sets, their fields, and unique identifiers (`@id`).
- Loaded records into pandas DataFrames for inspection.
- Conducted basic EDA including filtering, normalization, grouping, and visualization of key numeric and categorical fields.

For advanced use, you may:
- Explore other record sets or document sections (`@id`),
- Join information from multiple sets,
- Perform machine learning tasks using the tabular representations.

**Remember to always reference all entities by their `@id` fields in future analyses to ensure reproducibility and clarity.**